# ZNE simulations for best CVAR configurations

In this notebook a simulation of the best CVAR configuration for each MaxCut instance will be done. The ZNE configuration choosen for each instance will be based on the result of the prior work "$\href{https://ieeexplore.ieee.org/document/10821216}{\text{Characterizing the Effects of Zero-Noise Extrapolation on a QAOA Workflow}}$".

In [24]:
import networkx as nx
from openqaoa.problems import MaximumCut
from qiskit_aer.noise import (NoiseModel, depolarizing_error)
from openqaoa.backends import create_device
from openqaoa import QAOA  
from qiskit import transpile
from openqaoa.utilities import ground_state_hamiltonian

import os
import json

## Create Noise Model

In [2]:
one_qubit_gates = ['h','rx']
two_qubits_gates = ['rzz']

#create depol. noise
def add_depolarizing_error(noise_model,prob1, prob2):
    noise_model = add_one_qubit_depolarizing_error(noise_model,prob1)
    noise_model = add_two_qubits_depolarizing_error(noise_model,prob2)
    return noise_model

#create 1 qubit depol. noise
def add_one_qubit_depolarizing_error(noise_model,prob):
    error = depolarizing_error(prob, 1)
    noise_model.add_all_qubit_quantum_error(error,one_qubit_gates)
    return noise_model

#create 2 qubits depol.noise
def add_two_qubits_depolarizing_error(noise_model,prob):
    error = depolarizing_error(prob, 2)
    noise_model.add_all_qubit_quantum_error(error, two_qubits_gates)
    return noise_model

noise_model = add_depolarizing_error(NoiseModel(),0.0001989, 0.007905) #ibm_quebec, 19/01/2024

## Create MaxCut Problems

In [3]:
graph1 = nx.Graph()
graph1.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph1.add_edges_from([(0, 6), (0, 8), (1, 5), (2, 7), (2, 8), (3, 5), (3, 7), (4, 8), (6, 7), (7, 9), (8, 9)])

graph2 = nx.Graph()
graph2.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph2.add_edges_from([(0, 2), (0, 3), (0, 4), (0, 5), (0, 7), (0, 8), (1, 4), (1, 6), (1, 8), (1, 9), (2, 4), (2, 6), (3, 4), (3, 6), (3, 8), (3, 9), (4, 5), (4, 7), (4, 9), (5, 8), (6, 7), (7, 8), (7, 9), (8, 9)])

graph3 = nx.Graph()
graph3.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph3.add_edges_from([(0, 1), (0, 2), (0, 4), (0, 9), (1, 5), (1, 6), (1, 7), (1, 8), (2, 3), (2, 4), (2, 6), (3, 4), (3, 5), (3, 6), (3, 7), (3, 8), (3, 9), (4, 5), (4, 7), (4, 8), (5, 6), (5, 7), (5, 8), (5, 9), (6, 8), (6, 9), (7, 8), (7, 9)])

graph4 = nx.Graph()
graph4.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph4.add_edges_from([(0, 3), (1, 3), (1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (2, 3), (3, 4), (3, 6), (3, 7), (4, 5), (4, 6), (4, 9), (5, 6), (5, 7), (5, 8), (6, 7), (6, 8), (6, 9), (7, 9), (8, 9)])

graph5 = nx.Graph()
graph5.add_nodes_from([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
graph5.add_edges_from([(0, 5), (0, 7), (0, 8), (1, 2), (1, 3), (1, 4), (1, 6), (1, 9), (2, 3), (2, 4), (2, 5), (2, 8), (2, 9), (3, 5), (3, 6), (3, 7), (3, 9), (4, 6), (4, 7), (4, 9), (5, 6), (6, 7), (6, 8), (7, 9)])

mc1 = MaximumCut(graph1)
mc2 = MaximumCut(graph2)
mc3 = MaximumCut(graph3)
mc4 = MaximumCut(graph4)
mc5 = MaximumCut(graph5)

mcs = [mc1, mc2, mc3, mc4, mc5]

## Best CVAR configurations

In [31]:
# These cvar configuration parameters are the same in the 5 instances
OPTIMIZER='COBYLA'
PARAM_TYPE='standard'
INIT_TYPE='ramp'
MIXER_HAMILTONIAN='x'
general_cvar_conf = {
    'optimizer':OPTIMIZER,
    'param_type':PARAM_TYPE,
    'init_type':INIT_TYPE,
    'mixer_hamiltonian':MIXER_HAMILTONIAN
}

cvar_confs = [
    {   #instance 0
        'p':4,
        'cvar':0.5
    },
    {   #instance 1
        'p':4,
        'cvar':0.3
    },
    {   #instance 2
        'p':3,
        'cvar':0.5
    },
    {   #instance 3
        'p':3,
        'cvar':0.5
    },
    {   #instance 4
        'p':3,
        'cvar':0.5
    }
]

## Best ZNE configuration

The ZNE configuration was choosen considering the trade-off between approximation ratio and number of iterations. Since we will delimiting the number of iterations to $200$, we need a a ZNE configuration which achieves and acceptable aprox. ratio with not much number of iterations.

In this case, we choose Richardson with scale factors $1,2,3$.

In [5]:
zne_conf = {
    'factory':'Richardson',
    'scale_factors' : [1,2,3]
}

## Run experiment

In [34]:

# Run CVAR + ZNE for each tuple of data
for i,(mc,cvar_conf) in enumerate(zip(mcs,cvar_confs)):
    
    # get the MaxCut qubo
    mc = mc.qubo

    #####################################################################
    ###################### CVAR + ZNE ################################
    ##################################################################

    # create qiskit device
    qiskit_device = create_device(location='local', name='qiskit.shot_simulator')

    # set basic QAOA props
    q = QAOA()    
    q.set_device(qiskit_device)
    q.set_circuit_properties(p=cvar_conf['p'], param_type=PARAM_TYPE, init_type=INIT_TYPE, mixer_hamiltonian=MIXER_HAMILTONIAN)
    q.set_backend_properties(cvar_alpha=cvar_conf['cvar'],n_shots=5000,seed_simulator=1,noise_model=noise_model)
    q.set_classical_optimizer(method=OPTIMIZER, maxiter=200, tol=0.001, optimization_progress=True, cost_progress=True, parameter_log=True)

    # set ZNE props
    q.set_error_mitigation_properties(error_mitigation_technique='mitiq_zne',**zne_conf)#factory = 'Richardson', scale_factors = [1,2,3])

    # compile. It is necessary to transpile the parametric circuit with the same basis gates that has the noise model.
    q.compile(mc)
    q.backend.parametric_circuit = transpile(q.backend.parametric_circuit,basis_gates=["h","rx","rzz"])

    # optimize and get result
    print('optimizing instance'+str(i))
    q.optimize()
    correct_solution = ground_state_hamiltonian(q.cost_hamil)
    print(correct_solution)
    opt_results = q.result

    result = {
        'cvar_configuration':{**cvar_conf,**general_cvar_conf},
        'zne_configuration':zne_conf,
        'ground_state_energy':correct_solution[0],
        'qaoa_result':opt_results.asdict()
    }
    
    filename = 'results_cvar_qaoa/prob%s/mitCVAR/prob%s-cvar%s-p%s-%s-%s-%s-%s-%s_mitcvar.json'% (str(i),str(i),
                                                                             str(cvar_conf['cvar']),
                                                                             str(cvar_conf['p']),
                                                                             str(PARAM_TYPE),
                                                                             str(INIT_TYPE),
                                                                             str(MIXER_HAMILTONIAN),
                                                                             str(OPTIMIZER),
                                                                             str(zne_conf['factory']))    
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    with open(filename, 'w') as archivo:
        json.dump(result, archivo)

optimizing instance0
(-9.0, ['0000010110', '1000010110', '0000011110', '1111100001', '0111101001', '1111101001'])
optimizing instance1
(-16.0, ['0111010100', '1000101011'])
optimizing instance2
(-12.0, ['1001011100', '0110100011'])
optimizing instance3
(-12.0, ['1110100110', '0001011001'])
optimizing instance4
(-12.0, ['1011101000', '0100110110', '0101110110', '1010001001', '1011001001', '0100010111'])
